# Day 05 — Exception Handling, Logging & Retry
**Module 01: Python for AI Engineering**  
Vidya Sankalp | Applied GenAI Engineering

**Goal:** Learn how to handle errors safely, understand error messages, record useful logs, and retry an operation when a temporary problem occurs.

## Learning path
1. What is an exception?
2. `try` and `except`
3. Catching specific errors and reading the message with `as`
4. `finally` — code that always runs
5. `raise` — reporting invalid input
6. Logging with different levels
7. Handling a bad CSV row
8. A simple retry example

We'll begin with small examples and gradually connect them to real application scenarios. Custom exception classes and advanced retry patterns are outside today's scope.

---


## 1 — Imports

We'll import each module when we first need it. These modules are included with Python; you don't need to install them.

In [ ]:
import json
import logging
import time

---


## 2 — What is an Exception?

An **exception** is an error that happens while a program is running. For example, converting `"hello"` to an integer is not possible.

Without handling an exception, the program stops at that line. With `try/except`, we can respond to the problem and continue safely.

### Common exceptions

| Exception | Example | Meaning |
|---|---|---|
| `ValueError` | `int("hello")` | The value is not in the expected format |
| `TypeError` | `len(None)` | The value has the wrong type |
| `KeyError` | `customer["city"]` when `city` is missing | A dictionary key doesn't exist |
| `FileNotFoundError` | Opening a file that isn't there | The file couldn't be found |
| `json.JSONDecodeError` | `json.loads("{bad}")` | The text isn't valid JSON |

You don't need to memorize every exception. Start by reading the error name and message.

---


## 3 — `try` and `except`

Put code that might fail inside `try`. If the specified error happens, Python runs the matching `except` block.

```python
try:
    age = int("hello")
except ValueError:
    print("Please enter a whole number.")
```

**Important:** Catch the specific exception you expect, such as `ValueError`, rather than catching every possible error.

In [ ]:
# This conversion fails, so we handle the ValueError.
try:
    number = int("3.14")
    print(number)
except ValueError:
    print("That text is not a valid integer.")

In [ ]:
# Catch a ValueError when converting text to an integer.
def safe_to_int(value):
    try:
        return int(value)
    except ValueError:
        print(f"Cannot convert {value!r} to an integer.")
        return None

print(safe_to_int("1001"))
print(safe_to_int("hello"))

In [ ]:
# A dictionary lookup can raise KeyError; a missing or None name needs care.
def get_customer_name(customer, key):
    try:
        value = customer[key]
        return value.strip().title()
    except KeyError:
        print(f"Customer field {key!r} is missing.")
        return "Unknown"
    except AttributeError:
        print(f"Customer field {key!r} has no text value.")
        return "Unknown"

customer = {"name": "prudhvi akella", "email": None}
print(get_customer_name(customer, "name"))
print(get_customer_name(customer, "email"))
print(get_customer_name(customer, "city"))

In [ ]:
# Prefer specific exceptions over a broad `except Exception`.
# This function handles only the two errors we expect.
def get_customer_name_safe(customer, key):
    try:
        return customer[key].strip().title()
    except (KeyError, AttributeError):
        return "Unknown"

print(get_customer_name_safe({"name": "ravi kumar"}, "name"))
print(get_customer_name_safe({}, "name"))

In [ ]:
# One except block can handle more than one expected exception type.
def parse_number(value):
    try:
        return float(value)
    except (ValueError, TypeError):
        print(f"Cannot convert {value!r} to a number.")
        return None

print(parse_number("205.21"))
print(parse_number("hello"))
print(parse_number(None))

In [ ]:
# TypeError example, handled so the notebook can continue.
try:
    print(float(None))
except TypeError as error:
    print(f"Conversion failed: {error}")

---


## 4 — Reading the error message with `as`

Use `except SomeError as error` to capture the exception object. Its message can help you understand what went wrong.

```python
try:
    age = int("hello")
except ValueError as error:
    print(f"Error details: {error}")
```

In [ ]:
# json.loads() raises JSONDecodeError when the text is not valid JSON.
bad_json = '{"category": "TRACK_ORDER", missing_quote: "high"}'

try:
    parsed = json.loads(bad_json)
except json.JSONDecodeError as error:
    print(f"Could not read JSON: {error}")
    parsed = {}

print("Parsed value:", parsed)

---


## 5 — `finally`: code that always runs

The `finally` block runs whether the operation succeeds or an exception occurs. It is often used for cleanup or for recording that an operation has finished.

```python
try:
    print("Working...")
except ValueError:
    print("Something went wrong.")
finally:
    print("Finished.")
```

In [ ]:
# A tiny example: finally runs even when the conversion fails.
try:
    number = int("hello")
except ValueError:
    print("Conversion failed.")
finally:
    print("This message always appears.")

In [ ]:
def parse_json_safely(raw_text):
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError as error:
        print(f"Invalid JSON: {error.msg}")
        return {}
    finally:
        print("JSON parsing attempt finished.")

print(parse_json_safely('{"status": "ok"}'))
print(parse_json_safely('{bad json}'))

---


In [ ]:
# Same idea, written out step by step.
raw_response = '{bad json here}'

try:
    result = json.loads(raw_response)
except json.JSONDecodeError as error:
    print(f"Could not parse the response: {error.msg}")
    result = {}

print("Result:", result)

## 6 — `raise`: report invalid input

Use `raise` when your function detects a problem and cannot return a valid result. Python's built-in exceptions are enough for today's examples.

```python
def validate_age(age):
    if age < 0:
        raise ValueError("Age cannot be negative.")
    return age

print(validate_age(25))
```

When invalid input is passed, the caller can catch the `ValueError` and decide what to do.

In [ ]:
def validate_customer_id(customer_id):
    try:
        customer_id = int(customer_id)
    except (ValueError, TypeError) as error:
        raise ValueError("Customer ID must be a whole number.") from error

    if customer_id <= 0:
        raise ValueError("Customer ID must be greater than zero.")

    return customer_id

print(validate_customer_id("1001"))

for test_id in ["abc", "-5"]:
    try:
        print(validate_customer_id(test_id))
    except ValueError as error:
        print("Invalid ID:", error)

### Optional extension: how exceptions travel between functions

If a function raises an exception and doesn't catch it, Python passes it to the function that called it. This continues until some function handles it.

**Rule of thumb:** handle an error where you can make a useful decision. You don't need to study the full call-stack example to understand today's basics.

In [ ]:
def parse_customer_id(raw_id):
    customer_id = int(raw_id)
    if customer_id <= 0:
        raise ValueError("Customer ID must be greater than zero.")
    return customer_id

def build_customer_prompt(raw_id):
    try:
        customer_id = parse_customer_id(raw_id)
        return f"Prepare a prompt for customer {customer_id}."
    except ValueError as error:
        return f"Could not build prompt: {error}"

print(build_customer_prompt("1001"))
print(build_customer_prompt("abc"))

---


## 7 — The `logging` module

`print()` is useful while learning. In applications, **logging** lets us record messages with severity levels and filter which messages appear.

### Common log levels

| Level | When to use |
|---|---|
| `DEBUG` | Detailed information useful while developing |
| `INFO` | Normal events, such as starting a task |
| `WARNING` | Something unexpected happened, but the program can continue |
| `ERROR` | An operation failed |
| `CRITICAL` | A serious problem affecting the application |

A log level acts like a filter. For example, setting the level to `WARNING` hides `DEBUG` and `INFO` messages.

In [ ]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(levelname)s: %(message)s",
    force=True,
)
log = logging.getLogger("day05")

log.debug("Detailed information for debugging")
log.info("Program started")
log.warning("A value was missing; using a default")
log.error("An operation failed")
log.critical("A serious problem occurred")

In [ ]:
def safe_parse_llm_json(raw_response):
    log.info("Trying to parse the response.")

    try:
        result = json.loads(raw_response)
        log.info("Response parsed successfully.")
        return result
    except json.JSONDecodeError as error:
        log.error("Response was not valid JSON: %s", error.msg)
        return {}
    finally:
        log.info("Parsing attempt finished.")

print(safe_parse_llm_json('{"category": "TRACK_ORDER"}'))
print(safe_parse_llm_json('{bad json}'))

---


## 8 — Handling errors in CSV data

CSV rows often contain text, even for numbers. A row may also be missing a required column. We can catch these errors so one bad row doesn't stop the whole process.

In [ ]:
def parse_customer_csv_row(row):
    try:
        return {
            "customer_id": int(row["customer_id"]),
            "name": f'{row["first_name"]} {row["last_name"]}'.strip(),
            "email": row["email"].lower().strip(),
            "city": row["city"],
        }
    except KeyError as error:
        log.warning("Missing CSV column: %s", error)
        return {}
    except (ValueError, AttributeError) as error:
        log.warning("Invalid CSV value: %s", error)
        return {}

valid_row = {
    "customer_id": "1001", "first_name": "Ravi", "last_name": "Kumar",
    "email": "RAVI@example.com", "city": "Hyderabad"
}
missing_column = {"customer_id": "1002", "first_name": "Ananya"}
bad_id = {
    "customer_id": "not-a-number", "first_name": "Ananya",
    "last_name": "Sharma", "email": "a@example.com", "city": "Pune"
}

print(parse_customer_csv_row(valid_row))
print(parse_customer_csv_row(missing_column))
print(parse_customer_csv_row(bad_id))

---


## 9 — Retry when a temporary problem occurs

Some failures are temporary, such as a timeout or a short network interruption. A retry tries the operation again after waiting.

**Exponential backoff** means the wait gets longer after each failed attempt—for example, 0.1 seconds, then 0.2 seconds, then 0.4 seconds.

We'll simulate an API call so you can learn the retry pattern without connecting to a real service.

In [ ]:
def simulate_api_call(attempt):
    # Simulate two temporary failures, then a successful response.
    if attempt == 0:
        raise ConnectionError("Temporary connection problem")
    if attempt == 1:
        raise TimeoutError("The request took too long")
    return '{"status": "success"}'

In [ ]:
def call_with_retry(max_attempts=3):
    attempt = 0

    while attempt < max_attempts:
        try:
            log.info("Attempt %s of %s", attempt + 1, max_attempts)
            response = simulate_api_call(attempt)
            log.info("Request succeeded.")
            return response
        except (ConnectionError, TimeoutError) as error:
            attempt += 1
            if attempt == max_attempts:
                log.error("All attempts failed: %s", error)
                raise RuntimeError("Request failed after retries.") from error

            wait_seconds = 0.1 * (2 ** (attempt - 1))
            log.warning("Attempt failed: %s. Retrying in %.1f seconds.", error, wait_seconds)
            time.sleep(wait_seconds)

response = call_with_retry()
print("Final response:", response)

---


In [ ]:
# The retry demo above succeeds after two temporary failures.
# If every attempt failed, RuntimeError could be caught like this:
try:
    response = call_with_retry(max_attempts=1)
    print("Final response:", response)
except RuntimeError as error:
    print("Request could not be completed:", error)

## 10 — Filtering log levels

During development, `DEBUG` shows detailed messages. In a quieter run, set the level to `WARNING` to show only warnings and errors.

In [ ]:
logging.basicConfig(
    level=logging.WARNING,
    format="%(levelname)s: %(message)s",
    force=True,
)
log = logging.getLogger("day05")

log.debug("This debug message is hidden.")
log.info("This info message is hidden.")
log.warning("This warning is shown.")
log.error("This error is shown.")

# Restore the notebook's logging level for any later exercises.
logging.basicConfig(
    level=logging.DEBUG,
    format="%(levelname)s: %(message)s",
    force=True,
)
log = logging.getLogger("day05")

---


---
## 10 — Introduction to OpenTelemetry (optional)

So far, we've used **exceptions** to handle failures and **logging** to record what happened. In real applications—especially APIs and AI applications—we may also want to see how a request moves through different steps.

**OpenTelemetry (OTel)** is a standard for collecting observability data such as:

- **Traces:** the journey of one request through an application.
- **Spans:** individual steps within that journey, such as parsing a response or calling an API.
- **Attributes:** useful details attached to a span, such as the operation name or attempt number.

**How it relates to logging:**

- Logging records messages about events.
- Tracing shows the steps and time taken during a request.
- They work together; OpenTelemetry does not replace Python's `logging` module.

We'll keep this as an optional introduction because OpenTelemetry requires an additional package, unlike Python's built-in `logging` module.

### Step 1 — Install the OpenTelemetry packages

Run this installation cell only if you want to try the optional example. In a notebook, `%pip` installs packages into the environment used by the kernel.

```python
%pip install opentelemetry-api opentelemetry-sdk
```

For this beginner demo, spans will be printed to the notebook output. A real application can export telemetry to an observability backend.

In [ ]:
# Optional demo: create a trace and one span, then print the span to the output.
# Run the installation step above before running this cell.

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter

# Configure a tracer that prints finished spans to the console.
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("day05-demo")

def parse_response_with_trace(raw_response):
    with tracer.start_as_current_span("parse_json_response") as span:
        span.set_attribute("operation", "parse_json")

        try:
            result = json.loads(raw_response)
            span.set_attribute("parse.success", True)
            log.info("JSON response parsed successfully.")
            return result
        except json.JSONDecodeError as error:
            span.set_attribute("parse.success", False)
            span.record_exception(error)
            span.set_status(trace.Status(trace.StatusCode.ERROR, "Invalid JSON"))
            log.error("Could not parse JSON response.")
            return None

print(parse_response_with_trace('{"status": "ok"}'))
print(parse_response_with_trace('{bad json}'))

### What to notice

- `start_as_current_span(...)` creates a span for one operation.
- `set_attribute(...)` adds searchable context to the span.
- `record_exception(...)` attaches exception details to the span.
- `set_status(...)` marks the span as failed.
- The example still uses `logging`; the trace adds a view of the operation itself.

**Remember:** Start with exceptions and logging first. Treat this OpenTelemetry example as a preview of how production AI applications can be observed.

## 11 — Summary — Day 05

| Concept | Remember |
|---|---|
| Exception | An error that occurs while a program is running |
| `try/except` | Run risky code and handle an expected error |
| Specific exceptions | Catch the error types you expect, such as `ValueError` |
| `except ... as error` | Capture the error message |
| `finally` | Runs whether the operation succeeds or fails |
| `raise` | Signal that an input or operation is invalid |
| `logging` | Record events with severity levels |
| `DEBUG` / `INFO` | Development details and normal events |
| `WARNING` / `ERROR` | Unexpected situations and failures |
| CSV row handling | Catch missing columns and invalid values |
| Retry | Try again after a temporary failure |
| Exponential backoff | Increase the wait between failed attempts |

## Practice

1. Convert `"42"` to an integer inside `try/except`.
2. Handle a missing `"city"` key in a dictionary.
3. Write a function that raises `ValueError` when a price is negative.
4. Log a warning when a CSV row is missing a required field.
5. Explain why a timeout may be worth retrying, while invalid input usually isn't.

**Next:** Day 06 — Decorators and Higher-Order Functions.\n\n**Optional preview:** OpenTelemetry adds traces and spans to help observe application workflows.